## Cell 1: Environment Setup

In [ ]:
import warnings
import os
import tempfile
import sys
import traceback
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any
from dataclasses import dataclass, field
from collections import Counter
import re
import math

warnings.filterwarnings('ignore')

# Create isolated temp directory
TEMP_DIR = Path(tempfile.mkdtemp(prefix='interview_analyzer_'))
print(f'Workspace created: {TEMP_DIR}')

# Structured logging / status display
from IPython.display import display, HTML, clear_output

def show_status(message: str, level: str = 'info'):
    '''Thread-safe status display with color coding.'''
    colors = {
        'info': '#005599',
        'success': '#198754',
        'warning': '#ffc107',
        'error': '#dc3545',
        'analyze': '#6f42c1'
    }
    icon = {'info': 'ℹ️', 'success': '✅', 'warning': '⚠️', 'error': '❌', 'analyze': '🔬'}
    clear_output(wait=True)
    display(HTML(f'''
    <div style='
        background: rgba(0,85,153,0.08);
        border-left: 4px solid {colors.get(level, '#005599')};
        border-radius: 8px;
        padding: 12px 16px;
        margin: 8px 0;
        font-family: system-ui, sans-serif;
    '>
        <b>{icon.get(level, 'ℹ️')} {level.upper()}</b> — {message}
    </div>
    '''))

def show_error(exc: Exception, context: str = ''):
    '''Pretty-print exceptions without crashing the notebook.'''
    tb = traceback.format_exc()
    display(HTML(f'''
    <div style='
        background: #fff3f3;
        border-left: 4px solid #dc3545;
        border-radius: 8px;
        padding: 12px;
        margin: 8px 0;
    '>
        <b>❌ Error in {context}</b><br>
        <pre style='background:#f8f8f8;padding:8px;border-radius:4px;overflow-x:auto;'>{tb}</pre>
    </div>
    '''))
    print(f'[ERROR] {context}: {exc}')

## Cell 2: UI Styling

In [ ]:
from IPython.display import display, HTML

display(HTML('''
<style>
.glass-card {
    background: rgba(255,255,255,0.08);
    border-radius: 20px;
    padding: 20px;
    margin: 15px 0;
    backdrop-filter: blur(10px);
    border: 1px solid rgba(255,255,255,0.1);
}
.gradient-text {
    background: linear-gradient(45deg, #00c6ff, #0072ff);
    -webkit-background-clip: text;
    color: transparent;
    font-weight: bold;
}
.metric-box {
    display: inline-block;
    padding: 8px 16px;
    border-radius: 12px;
    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
    color: white;
    font-weight: 600;
    margin: 4px;
}
</style>
'''))

## Cell 3: Typed Data Structures

In [ ]:
@dataclass
class TranscriptResult:
    text: str
    duration_sec: float
    wpm: float
    segments: List[Dict] = field(default_factory=list)

@dataclass
class EmotionFrame:
    timestamp: float
    expression: str      # neutral, happy, surprise, focus, uncertain, etc.
    confidence: float
    raw_deepface: Dict   # keep raw for transparency

@dataclass
class GazeFrame:
    timestamp: float
    looking_at_camera: bool
    gaze_score: float    
    face_detected: bool
    reason: str          

@dataclass
class FillerStats:
    total_fillers: int
    filler_ratio: float
    per_filler_counts: Dict[str, int]
    severity: str
    false_positive_risk: str  # honest assessment

@dataclass
class VoiceFeatures:
    confidence_score: float
    confidence_label: str
    energy: float
    consistency: float
    speaking_ratio: float
    pause_quality: float
    pitch_variability: float
    expression_index: float
    steadiness: float
    clarity_proxy: float

@dataclass
class ContentQuality:
    clarity_score: float
    relevance_score: float
    structure_score: float
    tech_vocab_score: float
    content_quality: float
    keywords: List[str]
    avg_sentence_length: float

@dataclass
class InterviewReport:
    overall_score: float
    verdict: str
    communication: float
    confidence: float
    eye_contact_score: float
    content_quality: float
    emotion_stability: float
    speaking_pace: str
    strengths: List[str]
    improvements: List[str]
    transcript_snippet: str
    duration_sec: float
    total_words: int

## Cell 4: Upload Interface

In [ ]:
import ipywidgets as widgets
from IPython.display import display

video_path: Optional[Path] = None

upload_widget = widgets.FileUpload(
    accept='.mp4,.avi,.mov,.mkv,.webm',
    multiple=False,
    description='📁 Upload Video',
    layout=widgets.Layout(width='300px')
)

analyze_btn = widgets.Button(
    description='▶ Analyze Video',
    button_style='success',
    layout=widgets.Layout(width='200px', height='44px')
)

status_label = widgets.HTML('<i>Waiting for upload...</i>')

def on_upload_change(change):
    global video_path
    if upload_widget.value:
        uploaded = upload_widget.value[0]
        filename = uploaded['name']
        content = uploaded['content']
        video_path = TEMP_DIR / filename
        with open(video_path, 'wb') as f:
            f.write(content)
        status_label.value = f"<b style='color:#198754;'>✅ Uploaded:</b> {filename} ({len(content)/1024/1024:.1f} MB)"
        print(f'Saved to: {video_path}')

upload_widget.observe(on_upload_change, names='value')

display(widgets.VBox([
    widgets.HTML("<h2 class='gradient-text'>🎥 Interview Analyzer v2.0</h2>"),
    widgets.HTML("<p style='color:#666;'>Upload a video of yourself answering interview questions. Max 5 min recommended.</p>"),
    upload_widget,
    analyze_btn,
    status_label
]))

## Cell 5: Audio Extraction 


In [ ]:
from moviepy import VideoFileClip

def extract_audio(video_path: Path) -> Optional[Path]:
    '''Extract the audio to WAV. Returns None if video has no audio.'''
    try:
        audio_path = TEMP_DIR / 'audio_extracted.wav'
        with VideoFileClip(str(video_path)) as clip:
            if clip.audio is None:
                show_status('Video has no audio stream — continuing with visual-only analysis.', 'warning')
                return None
            clip.audio.write_audiofile(str(audio_path), logger=None, fps=16000, nbytes=2, codec='pcm_s16le')
        return audio_path
    except Exception as e:
        show_error(e, 'Audio Extraction')
        return None

## Cell 6: Frame Sampling with Validation

In [ ]:
import cv2
import numpy as np

def sample_frames(video_path: Path, target_fps: float = 0.5, max_frames: int = 300) -> Tuple[List[np.ndarray], List[float]]:
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise ValueError(f'Cannot open video: {video_path}')
    
    original_fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    if original_fps <= 0:
        original_fps = 30.0  # fallback
    
    interval = max(1, int(original_fps / target_fps))
    estimated = total_frames // interval
    
    if estimated > max_frames:
        interval = max(1, total_frames // max_frames)
        show_status(f'Large video detected. Sampling every {interval} frames (≈{max_frames} frames max).', 'warning')
    
    frames, timestamps = [], []
    count = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if count % interval == 0:
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(rgb)
            timestamps.append(count / original_fps)
        count += 1
    cap.release()
    
    if not frames:
        raise ValueError('No frames could be extracted from video.')
    
    print(f'Extracted {len(frames)} frames @ ~{target_fps:.1f} fps')
    return frames, timestamps

## Cell 7: Transcription with Word-Level Timestamps

In [ ]:
from faster_whisper import WhisperModel

# Load once — tiny for speed, medium for accuracy if RAM allows(16GB or more)
whisper_model = WhisperModel('tiny', device='cpu', compute_type='int8')

def transcribe_audio(audio_path: Path) -> TranscriptResult:
    segments, info = whisper_model.transcribe(str(audio_path), word_timestamps=True)
    
    seg_list = []
    all_words = []
    for seg in segments:
        seg_list.append({
            'start': seg.start,
            'end': seg.end,
            'text': seg.text
        })
        if seg.words:
            all_words.extend([w.word for w in seg.words])
    
    transcript = ' '.join(all_words) if all_words else ' '.join(s['text'] for s in seg_list)
    duration = info.duration if info.duration > 0 else 1.0
    total_words = len(transcript.split())
    wpm = (total_words / duration) * 60.0
    
    return TranscriptResult(
        text=transcript,
        duration_sec=duration,
        wpm=round(wpm, 1),
        segments=seg_list
    )

## Cell 8: Gaze Direction Analysis 

In [ ]:
import mediapipe as mp
import numpy as np
from typing import Tuple, List
import os
import urllib.request


# Download Face Landmarker model (one-time setup)
MODEL_PATH = 'face_landmarker.task'
MODEL_URL = 'https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task'

if not os.path.exists(MODEL_PATH):
    print('Downloading MediaPipe Face Landmarker model...')
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    print(f'Model saved: {MODEL_PATH}')
else:
    print(f'Model already exists: {MODEL_PATH}')


BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=VisionRunningMode.VIDEO,
    num_faces=1,
    min_face_detection_confidence=0.5,
    min_face_presence_confidence=0.5,
    min_tracking_confidence=0.5,
    output_face_blendshapes=False,
    output_facial_transformation_matrixes=False
)

landmarker = FaceLandmarker.create_from_options(options)

LEFT_EYE_CORNERS = [33, 133]
RIGHT_EYE_CORNERS = [362, 263]
LEFT_IRIS = [468, 469, 470, 471]
RIGHT_IRIS = [473, 474, 475, 476]

def get_iris_center(landmarks, indices, w, h):
    xs = [landmarks[i].x * w for i in indices]
    ys = [landmarks[i].y * h for i in indices]
    return np.mean(xs), np.mean(ys)

def get_eye_center(landmarks, indices, w, h):
    xs = [landmarks[i].x * w for i in indices]
    ys = [landmarks[i].y * h for i in indices]
    return np.mean(xs), np.mean(ys)

def estimate_gaze_score(frame_rgb: np.ndarray, frame_timestamp_ms: int = 0) -> Tuple[float, bool, str]:
    h, w = frame_rgb.shape[:2]
    
    # Convert numpy array to MediaPipe Image
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
    
    # Detect landmarks 
    results = landmarker.detect_for_video(mp_image, frame_timestamp_ms)
    
    if not results.face_landmarks:
        return 0.0, False, 'No face detected'
    
    landmarks = results.face_landmarks[0]
    
    left_ix, left_iy = get_iris_center(landmarks, LEFT_IRIS, w, h)
    right_ix, right_iy = get_iris_center(landmarks, RIGHT_IRIS, w, h)
    
    left_ex, left_ey = get_eye_center(landmarks, LEFT_EYE_CORNERS, w, h)
    right_ex, right_ey = get_eye_center(landmarks, RIGHT_EYE_CORNERS, w, h)
    
    left_eye_width = abs(landmarks[LEFT_EYE_CORNERS[0]].x - landmarks[LEFT_EYE_CORNERS[1]].x) * w
    right_eye_width = abs(landmarks[RIGHT_EYE_CORNERS[0]].x - landmarks[RIGHT_EYE_CORNERS[1]].x) * w
    
    if left_eye_width < 1 or right_eye_width < 1:
        return 0.0, False, 'Face too small or too far'
    
    left_offset = abs(left_ix - left_ex) / left_eye_width
    right_offset = abs(right_ix - right_ex) / right_eye_width
    
    avg_offset = (left_offset + right_offset) / 2.0
    gaze_score = max(0.0, 1.0 - (avg_offset / 0.3))
    looking = gaze_score > 0.6
    
    reason = 'Looking at camera' if looking else 'Gaze appears off-center'
    return round(gaze_score, 3), looking, reason

def analyze_gaze(frames: List[np.ndarray], timestamps: List[float]) -> Tuple[List[GazeFrame], float]:
    gaze_frames = []
    for idx, frame in enumerate(frames):
        timestamp_ms = int(timestamps[idx] * 1000)
        score, looking, reason = estimate_gaze_score(frame, timestamp_ms)
        gaze_frames.append(GazeFrame(
            timestamp=timestamps[idx],
            looking_at_camera=looking,
            gaze_score=score,
            face_detected=(score > 0 or reason != 'No face detected'),
            reason=reason
        ))
    
    valid = [g for g in gaze_frames if g.face_detected]
    avg_score = np.mean([g.gaze_score for g in valid]) if valid else 0.0
    return gaze_frames, round(avg_score, 3)

## Cell 9: Facial Expression Analysis 


In [ ]:
from deepface import DeepFace
from tqdm.notebook import tqdm


EXPRESSION_MAP = {
    'neutral': 'composed',
    'happy': 'positive_expressive',
    'surprise': 'attentive',
    'sad': 'low_energy',
    'angry': 'intense',
    'fear': 'anxious',
    'disgust': 'negative_reaction'
}

def analyze_expressions(frames: List[np.ndarray], timestamps: List[float]) -> List[EmotionFrame]:
    '''Run DeepFace with confidence thresholding and remapping.'''
    emotions = []
    for idx, frame in enumerate(tqdm(frames, desc='Expression Analysis')):
        try:
            result = DeepFace.analyze(
                frame,
                actions=['emotion'],
                enforce_detection=False,
                silent=True
            )
            raw = result[0]['emotion']
            dominant = result[0]['dominant_emotion']
            confidence = raw[dominant] / 100.0
            
            # Only trust if confidence > 60%, else mark uncertain
            if confidence < 0.6:
                mapped = 'uncertain_reading'
            else:
                mapped = EXPRESSION_MAP.get(dominant, 'unknown')
            
            emotions.append(EmotionFrame(
                timestamp=timestamps[idx],
                expression=mapped,
                confidence=round(confidence, 3),
                raw_deepface=raw
            ))
        except Exception:
            emotions.append(EmotionFrame(
                timestamp=timestamps[idx],
                expression='no_detection',
                confidence=0.0,
                raw_deepface={}
            ))
    return emotions

## Cell 10: Filler Word Detection 


In [ ]:
FILLER_WORDS = [
    'um', 'uh', 'ah', 'er', 'hmm',
    'like', 'basically', 'literally', 'actually', 'honestly',
    'seriously', 'totally', 'completely', 'virtually', 'essentially',
    'you know', 'i mean', 'sort of', 'kind of',
    'so', 'well', 'right', 'okay', 'ok', 'yeah',
    'to be honest', 'truth be told', 'at the end of the day',
    'or something', 'or whatever', 'and stuff', 'and things'
]

def detect_fillers(transcript: str) -> FillerStats:
    text_lower = transcript.lower()
    words_total = len(text_lower.split())
    words_total = max(words_total, 1)
    
    per_filler = {}
    masked_text = text_lower
    
    for filler in sorted(FILLER_WORDS, key=len, reverse=True):
        if ' ' in filler:
            pattern = re.escape(filler)
        else:
            pattern = r'\b' + re.escape(filler) + r'\b'
        
        matches = list(re.finditer(pattern, masked_text))
        count = len(matches)
        if count > 0:
            per_filler[filler] = count
            # Mask out matched regions to prevent overlapping counts
            for m in reversed(matches):
                start, end = m.span()
                masked_text = masked_text[:start] + ' ' * (end - start) + masked_text[end:]
    
    total_fillers = sum(per_filler.values())
    ratio = (total_fillers / words_total) * 100.0
    
    if ratio <= 2.0:
        severity = 'Excellent'
    elif ratio <= 5.0:
        severity = 'Good'
    elif ratio <= 8.0:
        severity = 'Moderate'
    else:
        severity = 'Needs Improvement'
    
    # Honest risk assessment
    if ratio > 3.0:
        risk = 'High false-negative risk: fast speech, accents, or non-English words may be misclassified.'
    else:
        risk = 'Low false-positive risk detected.'
    
    return FillerStats(
        total_fillers=total_fillers,
        filler_ratio=round(ratio, 2),
        per_filler_counts=per_filler,
        severity=severity,
        false_positive_risk=risk
    )

## Cell 11: Voice Feature Extraction 


In [ ]:
import librosa
import numpy as np

def analyze_voice(audio_path: Path) -> VoiceFeatures:
    try:
        y, sr = librosa.load(str(audio_path), sr=None, mono=True)
        duration = len(y) / sr
        
        if duration < 3.0:
            show_status('Audio very short (<3s) — voice metrics unreliable.', 'warning')
        
        # Energy
        rms = librosa.feature.rms(y=y)[0]
        energy = float(np.mean(rms)) * 1000
        energy = min(100.0, energy)
        
        # Consistency 
        rms_mean = np.mean(rms) + 1e-9
        consistency = max(0.0, 100.0 - (np.std(rms) / rms_mean) * 100.0)
        
        # Speaking vs silence
        intervals = librosa.effects.split(y, top_db=25)
        speak_time = sum(e - s for s, e in intervals) / sr
        speaking_ratio = min(100.0, (speak_time / duration) * 100.0) if duration > 0 else 0.0
        
        # Pauses 
        num_pauses = max(0, len(intervals) - 1)
        pause_rate = num_pauses / duration if duration > 0 else 0
        pause_quality = max(0.0, 100.0 - pause_rate * 15.0)
        
        # Pitch variability 
        pitches, mags = librosa.piptrack(y=y, sr=sr)
        pitch_vals = pitches[mags > np.median(mags) + 1e-6]
        pitch_vals = pitch_vals[pitch_vals > 50]
        pitch_std = float(np.std(pitch_vals)) if len(pitch_vals) > 10 else 0.0
        
        if 10 <= pitch_std <= 120:
            pitch_var = min(100.0, pitch_std * 0.8)
        elif pitch_std < 10:
            pitch_var = 20.0  
        else:
            pitch_var = 60.0   
        
        # MFCC expression index 
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        expression_index = min(100.0, float(np.mean(np.std(mfccs, axis=1))) * 3.0)
        
        # Steadiness (ZCR stability)
        zcr = librosa.feature.zero_crossing_rate(y)[0]
        steadiness = max(0.0, 100.0 - float(np.std(zcr)) * 2000.0)
        
        # Clarity proxy via spectral centroid
        spec_cent = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
        spec_mean = float(np.mean(spec_cent))
        if 1000 <= spec_mean <= 3000:
            clarity = 100.0
        elif 500 <= spec_mean <= 4000:
            clarity = 75.0
        else:
            clarity = 45.0
        
        weights = {
            'energy': 0.15,
            'consistency': 0.10,
            'speaking': 0.15,
            'pauses': 0.10,
            'pitch': 0.15,
            'expression': 0.10,
            'steadiness': 0.10,
            'clarity': 0.15
        }
        scores = {
            'energy': energy,
            'consistency': consistency,
            'speaking': speaking_ratio,
            'pauses': pause_quality,
            'pitch': pitch_var,
            'expression': expression_index,
            'steadiness': steadiness,
            'clarity': clarity
        }
        
        score = sum(scores[k] * weights[k] for k in weights)
        score = max(0.0, min(100.0, score))
        
        if score >= 80:
            label = 'High Vocal Presence'
        elif score >= 60:
            label = 'Moderate Vocal Presence'
        elif score >= 40:
            label = 'Low Vocal Presence'
        else:
            label = 'Very Low Vocal Presence'
        
        return VoiceFeatures(
            confidence_score=round(score, 1),
            confidence_label=label,
            energy=round(energy, 1),
            consistency=round(consistency, 1),
            speaking_ratio=round(speaking_ratio, 1),
            pause_quality=round(pause_quality, 1),
            pitch_variability=round(pitch_var, 1),
            expression_index=round(expression_index, 1),
            steadiness=round(steadiness, 1),
            clarity_proxy=round(clarity, 1)
        )
    
    except Exception as e:
        show_error(e, 'Voice Analysis')
        return VoiceFeatures(
            confidence_score=50.0,
            confidence_label='Analysis Failed — Defaulted',
            energy=0.0, consistency=0.0, speaking_ratio=0.0,
            pause_quality=0.0, pitch_variability=0.0,
            expression_index=0.0, steadiness=0.0, clarity_proxy=0.0
        )

## Cell 12: Sentiment Analysis

In [ ]:
from transformers import pipeline

sentiment_pipeline = pipeline(
    'sentiment-analysis',
    model='distilbert-base-uncased-finetuned-sst-2-english',
    device=-1  # CPU
)

def analyze_sentiment(text: str) -> Dict[str, Any]:
    snippet = text[:512]
    result = sentiment_pipeline(snippet)[0]
    return {
        'label': result['label'],
        'confidence': round(result['score'], 3),
        'note': 'Based on first 512 chars only. Long-form context ignored.'
    }

## Cell 13: Content Quality (Keyword + Structure)


In [ ]:
TECH_KEYWORDS = [
    'machine learning', 'deep learning', 'tensorflow', 'pytorch', 'keras', 'opencv', 'python', 'java', 'c++', 'javascript', 
    'react', 'node.js', 'django', 'flask', 'fastapi', 'cnn', 'rnn', 'lstm', 'transformer', 'bert', 'gpt', 'yolo', 'resnet', 
    'nlp', 'computer vision', 'data science', 'sql', 'nosql', 'mongodb', 'docker', 'kubernetes', 'aws', 'azure', 'gcp', 'git', 
    'ci/cd', 'devops', 'mlops', 'api', 'rest', 'graphql', 'microservices', 'neural network', 'regression', 'classification', 
    'clustering', 'reinforcement learning', 'pandas', 'numpy', 'scikit-learn', 'spark', 'hadoop', 'accuracy', 'precision', 
    'recall', 'f1', 'overfitting', 'cross-validation', 'rag', 'llm', 'vector database', 'embedding', 'fine-tuning', 'lora', 
    'qlora', 'langchain', 'agent', 'multimodal', 'diffusion', 'sam', 'vision transformer', 'mamba', 'state space model', 
    'quantization', 'onnx', 'tensorrt', 'triton', 'feature store', 'data drift', 'model registry', 'terraform', 'pulumi', 
    'helm', 'istio', 'kafka', 'airflow', 'dbt', 'snowflake', 'bigquery', 'redshift', 'clickhouse', 'duckdb', 'bm25', 'tf-idf', 
    'approximate nearest neighbor', 'hnsw', 'faiss', 'pinecone', 'milvus', 'chromadb', 'weaviate', 'pgvector', 'semantic search', 
    'shap', 'lime', 'explainable ai', 'federated learning', 'differential privacy', 'prompt engineering', 'chain of thought', 
    'tree of thoughts', 'jailbreak', 'hallucination', 'toxicity', 'bias mitigation', 'fairness', 'model card', 'a/b testing', 
    'multi-armed bandit', 'causal inference', 'propensity score', 'time series', 'arima', 'prophet', 'forecasting', 'neo4j', 
    'anomaly detection', 'fraud detection', 'recommendation system', 'collaborative filtering', 'graph neural network', 
    'gcn', 'gat', 'graphsage', 'pytorch geometric', 'dgl', 'networkx', 'knowledge graph', 'reasoning', 'cuda', 'tensor core', 
    'mixed precision', 'fp16', 'bf16', 'int8', 'distributed training', 'data parallel', 'model parallel', 'pipeline parallel', 
    'fsdp', 'deepspeed', 'megatron', 'gradient accumulation', 'checkpointing', 'edge ai', 'tinyml', 'tensorflow lite', 'ros2', 
    'coreml', 'onnx runtime', 'jetson', 'ros', 'slam', 'object detection', 'segmentation', 'pose estimation', 'optical flow', 
    'nerf', 'gaussian splatting', 'typescript', 'rust', 'go', 'kotlin', 'swift', 'html5', 'css3', 'vue', 'next.js', 'angular', 
    'spring boot', 'express', 'postgresql', 'redis', 'elasticsearch', 'cassandra', 'dynamodb', 'supabase', 'firebase', 'graphql', 
    'serverless', 'lambda', 'github actions', 'jenkins', 'ansible', 'prometheus', 'grafana', 'elk stack', 'linux', 'bash', 
    'webassembly', 'webgl', 'three.js', 'tailwind', 'bootstrap', 'sass', 'webpack', 'vite', 'npm', 'pnpm', 'yarn', 'deno', 'bun', 
    'restful', 'grpc', 'websockets', 'oauth', 'jwt', 'saml', 'ssl/tls', 'cybersecurity', 'cryptography', 'blockchain', 'solidity', 
    'ethereum', 'smart contract', 'web3', 'supercomputing', 'hpc', 'openmp', 'mpi', 'opencl', 'vulkan', 'directx', 'unreal engine', 
    'unity', 'blender', 'cuda programming', 'fpga', 'asic', 'hardware acceleration', 'embedded systems', 'rtos', 'arduino', 
    'raspberry pi', 'firmware', 'compiler', 'interpreter', 'llvm', 'ast', 'regex', 'json', 'yaml', 'xml', 'parquet', 'avro', 
    'orc', 'data lake', 'data warehouse', 'lakehouse', 'databricks', 'iceberg', 'delta lake', 'etl', 'elt', 'data pipeline', 
    'fivetran', 'stitch', 'airbyte', 'presto', 'trino', 'athena', 'power bi', 'tableau', 'looker', 'superset', 'metabase', 
    'data visualization', 'matplotlib', 'seaborn', 'plotly', 'bokeh', 'd3.js', 'statistics', 'linear algebra', 'calculus', 
    'probability', 'hypothesis testing', 'p-value', 'anova', 'chi-square', 'bayes theorem', 'stochastic process', 'optimization', 
    'gradient descent', 'adam', 'sgd', 'backpropagation', 'activation function', 'relu', 'sigmoid', 'tanh', 'softmax', 'loss function', 
    'mse', 'cross entropy', 'regularization', 'l1/l2', 'dropout', 'batch normalization', 'layer normalization', 'attention mechanism', 
    'self-attention', 'multi-head attention', 'positional encoding', 'tokenization', 'word2vec', 'glove', 'fasttext', 'sentence transformers'
]

def analyze_content_quality(transcript: str) -> ContentQuality:
    text_lower = transcript.lower()
    words = transcript.split()
    sentences = [s.strip() for s in re.split(r'[.!?]+', transcript.strip()) if s.strip()]
    
    num_words = max(len(words), 1)
    num_sent = max(len(sentences), 1)
    avg_sent_len = num_words / num_sent
    
    # Clarity: ideal sentence length 10-20 words
    if 10 <= avg_sent_len <= 20:
        clarity = 90.0
    elif 7 <= avg_sent_len <= 25:
        clarity = 70.0
    else:
        clarity = 45.0
    
    # Technical keywords — unique matches only, capped density
    detected = []
    for kw in TECH_KEYWORDS:
        if kw in text_lower:
            detected.append(kw.title())
    unique_kw = list(dict.fromkeys(detected))  
    
    # Cap density to prevent keyword stuffing reward
    raw_density = len(unique_kw) / (num_words / 100.0)
    capped_density = min(raw_density, 8.0)  # max 8 keywords per 100 words counted
    relevance = min(100.0, capped_density * 12.0)
    
    structure_markers = [
        'first', 'second', 'third', 'finally', 'in conclusion',
        'for example', 'for instance', 'such as', 'because', 'therefore',
        'however', 'additionally', 'furthermore', 'i have', 'i worked',
        'i built', 'i developed', 'my project', 'our team', 'the result',
        'the outcome', 'led to', 'achieved', 'improved', 'increased', 'decreased'
    ]
    structure_hits = sum(1 for m in structure_markers if m in text_lower)
    structure = min(100.0, structure_hits * 8.0 + 20.0)
    
    # Tech vocab score 
    tech_vocab = min(100.0, len(unique_kw) * 6.0)
    
    content_quality = (
        clarity * 0.25 +
        relevance * 0.30 +
        structure * 0.25 +
        tech_vocab * 0.20
    )
    
    return ContentQuality(
        clarity_score=round(clarity),
        relevance_score=round(relevance),
        structure_score=round(structure),
        tech_vocab_score=round(tech_vocab),
        content_quality=round(content_quality, 1),
        keywords=unique_kw[:20],  # cap output list
        avg_sentence_length=round(avg_sent_len, 1)
    )

## Cell 14: Scoring Engine 


In [ ]:
def compute_scores(
    transcript: TranscriptResult,
    gaze_frames: List[GazeFrame],
    voice: VoiceFeatures,
    filler: FillerStats,
    content: ContentQuality,
    emotions: List[EmotionFrame]
) -> InterviewReport:
    
    #  Speaking Pace 
    wpm = transcript.wpm
    if wpm < 100:
        pace = 'Too Slow'
        pace_note = 'Below ideal range. Risk of losing interviewer engagement.'
    elif wpm <= 160:
        pace = 'Ideal'
        pace_note = 'Comfortable pace for professional communication.'
    else:
        pace = 'Too Fast'
        pace_note = 'Above ideal range. Key points may be missed.'
    
    # Normalize pace to 0-100 score (peak at 135 WPM)
    wpm_score = max(0.0, 100.0 - abs(wpm - 135.0) * 1.5)
    
    #  Communication 
    filler_penalty = min(filler.filler_ratio * 8.0, 100.0)
    filler_score = max(0.0, 100.0 - filler_penalty)
    communication = wpm_score * 0.45 + filler_score * 0.55
    
    #  Eye Contact 
    valid_gaze = [g for g in gaze_frames if g.face_detected]
    if valid_gaze:
        avg_gaze = np.mean([g.gaze_score for g in valid_gaze])
        eye_pct = (sum(1 for g in valid_gaze if g.looking_at_camera) / len(valid_gaze)) * 100.0
    else:
        avg_gaze = 0.0
        eye_pct = 0.0
    
    eye_contact_score = min(100.0, eye_pct * 1.2)
    
    if eye_pct >= 75:
        eye_assessment = 'Strong camera engagement detected.'
    elif eye_pct >= 50:
        eye_assessment = 'Moderate camera engagement — consistent in most segments.'
    elif eye_pct >= 30:
        eye_assessment = 'Intermittent camera engagement — consider practicing direct-to-lens speaking.'
    else:
        eye_assessment = 'Low camera engagement — notes or second monitor may be distracting.'
    
    #  Confidence 
    emotion_positive = sum(1 for e in emotions if e.expression in ['composed', 'positive_expressive', 'attentive'])
    emotion_total = max(len([e for e in emotions if e.expression != 'no_detection']), 1)
    emotion_pos_pct = (emotion_positive / emotion_total) * 100.0
    
    confidence = (
        voice.confidence_score * 0.40 +
        eye_contact_score * 0.30 +
        emotion_pos_pct * 0.30
    )
    
    #  Emotion Stability 
    expression_counts = Counter(e.expression for e in emotions if e.expression != 'no_detection')
    total_exp = max(sum(expression_counts.values()), 1)
    negative_exp = expression_counts.get('low_energy', 0) + expression_counts.get('anxious', 0) + expression_counts.get('intense', 0) + expression_counts.get('negative_reaction', 0)
    positive_exp = expression_counts.get('composed', 0) + expression_counts.get('positive_expressive', 0) + expression_counts.get('attentive', 0)
    
    stability = min(100.0, max(0.0, (positive_exp / total_exp) * 100.0 * 0.8 - (negative_exp / total_exp) * 100.0 * 0.5 + 20.0))
    
    if stability >= 80:
        stability_label = 'Excellent'
    elif stability >= 60:
        stability_label = 'Good'
    else:
        stability_label = 'Needs Improvement'
    
    #  Overall 
    overall = (
        communication * 0.25 +
        confidence * 0.25 +
        content.content_quality * 0.30 +
        eye_contact_score * 0.10 +
        stability * 0.10
    )
    
    if overall >= 90:
        verdict = 'Excellent'
    elif overall >= 75:
        verdict = 'Good'
    elif overall >= 60:
        verdict = 'Average'
    else:
        verdict = 'Needs Improvement'
    
    #  Strengths / Improvements 
    strengths, improvements = [], []
    
    if pace == 'Ideal':
        strengths.append('Speaking pace is in the ideal professional range.')
    else:
        improvements.append(f'Speaking pace: {pace}. {pace_note}')
    
    if filler.severity in ('Excellent', 'Good'):
        strengths.append('Minimal filler word usage.')
    else:
        improvements.append(f'Filler word ratio is {filler.filler_ratio}%. Practice pausing silently instead.')
    
    if eye_contact_score >= 70:
        strengths.append('Good camera engagement / gaze alignment.')
    else:
        improvements.append(eye_assessment)
    
    if content.tech_vocab_score >= 50:
        strengths.append('Strong technical vocabulary demonstrated.')
    else:
        improvements.append('Use more specific technical terminology relevant to your domain.')
    
    if confidence >= 65:
        strengths.append('Vocal and visual signals project confidence.')
    else:
        improvements.append('Vocal energy and pitch variability suggest room for more expressive delivery.')
    
    if content.structure_score >= 50:
        strengths.append('Communication appears structured (markers like first, for example detected).')
    else:
        improvements.append('Structure answers using frameworks like STAR (Situation, Task, Action, Result).')
    
    if stability_label in ('Excellent', 'Good'):
        strengths.append('Stable facial composure throughout the interview.')
    else:
        improvements.append('Practice mock interviews to build comfort and reduce anxious expressions.')
    
    if len(content.keywords) >= 5:
        strengths.append(f'Demonstrated awareness of {len(content.keywords)} technical concepts.')
    else:
        improvements.append('Provide concrete technical examples and project references.')
    
    return InterviewReport(
        overall_score=round(overall, 1),
        verdict=verdict,
        communication=round(communication, 1),
        confidence=round(confidence, 1),
        eye_contact_score=round(eye_contact_score, 1),
        content_quality=round(content.content_quality, 1),
        emotion_stability=round(stability, 1),
        speaking_pace=pace,
        strengths=strengths,
        improvements=improvements,
        transcript_snippet=transcript.text[:2000],
        duration_sec=transcript.duration_sec,
        total_words=len(transcript.text.split())
    )

## Cell 15: Professional PDF Generation

In [ ]:
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, HRFlowable, PageBreak
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.pagesizes import letter
from reportlab.lib import colors
from reportlab.lib.units import inch
from datetime import datetime

def generate_pdf(report: InterviewReport, gaze_frames: List[GazeFrame], emotions: List[EmotionFrame], 
                 filler: FillerStats, content: ContentQuality, voice: VoiceFeatures,
                 sentiment: Dict, output_path: Optional[Path] = None) -> Path:
    
    if output_path is None:
        output_path = TEMP_DIR / 'Interview_Analysis_Report_v2.pdf'
    
    doc = SimpleDocTemplate(str(output_path), pagesize=letter,
                            leftMargin=0.75*inch, rightMargin=0.75*inch,
                            topMargin=0.75*inch, bottomMargin=0.75*inch)
    styles = getSampleStyleSheet()
    story = []
    
    # Custom styles
    h1 = ParagraphStyle('H1', parent=styles['Heading1'], fontSize=20, spaceAfter=8, textColor=colors.HexColor('#0d1b2a'))
    h2 = ParagraphStyle('H2', parent=styles['Heading2'], fontSize=14, spaceAfter=6, textColor=colors.HexColor('#1b4965'))
    h3 = ParagraphStyle('H3', parent=styles['Heading3'], fontSize=11, spaceAfter=4, textColor=colors.HexColor('#5fa8d3'))
    body = styles['Normal']
    small = ParagraphStyle('small', parent=body, fontSize=9, textColor=colors.grey)
    warning = ParagraphStyle('warning', parent=body, fontSize=10, textColor=colors.HexColor('#b08968'), backColor=colors.HexColor('#fff3e0'))
    
    def section(title):
        story.append(Spacer(1, 12))
        story.append(HRFlowable(width='100%', thickness=1.5, color=colors.HexColor('#1b4965')))
        story.append(Paragraph(title, h2))
    
    def kv(key, value):
        story.append(Paragraph(f'<b>{key}:</b> {value}', body))
    
    #  HEADER 
    story.append(Paragraph('🎯 AI Interview Analysis Report', h1))
    story.append(Paragraph(f'Generated: {datetime.now().strftime("%Y-%m-%d %H:%M")} · Analyzer v2.0', small))
    story.append(Spacer(1, 8))
    
    # Disclaimer box
    story.append(Paragraph(
        '<b>⚠️ IMPORTANT DISCLAIMER:</b> All scores are heuristic estimates based on algorithmic '
        'feature extraction. They do not measure intelligence, competence, or hireability. '
        'Use for self-improvement only.',
        warning
    ))
    story.append(Spacer(1, 12))
    
    #  1. SUMMARY 
    section('1. Executive Summary')
    dur_min = int(report.duration_sec // 60)
    dur_sec = int(report.duration_sec % 60)
    kv('Interview Duration', f'{dur_min}m {dur_sec}s')
    kv('Total Words Spoken', report.total_words)
    kv('Speaking Pace', f"{report.speaking_pace} ({report.total_words / max(report.duration_sec/60, 0.1):.1f} WPM)")
    kv('Overall Score', f'{report.overall_score:.1f} / 100')
    kv('Verdict', f'<b>{report.verdict}</b>')
    story.append(Spacer(1, 6))
    
    #  2. SCORE TABLE 
    section('2. Score Breakdown')
    score_data = [
        ['Category', 'Score', 'Weight', 'Notes'],
        ['Communication', f"{report.communication:.1f}", '25%', 'Pace + filler words'],
        ['Confidence', f"{report.confidence:.1f}", '25%', 'Voice + gaze + expression'],
        ['Content Quality', f"{report.content_quality:.1f}", '30%', 'Clarity + structure + keywords'],
        ['Eye Contact', f"{report.eye_contact_score:.1f}", '10%', 'Gaze alignment via face mesh'],
        ['Emotion Stability', f"{report.emotion_stability:.1f}", '10%', 'Expression consistency'],
        ['OVERALL', f'{report.overall_score:.1f}', '100%', f'Verdict: {report.verdict}'],
    ]
    
    def score_color(val):
        try:
            v = float(val)
            return (colors.HexColor('#d4edda') if v >= 75 else
                    colors.HexColor('#fff3cd') if v >= 60 else
                    colors.HexColor('#f8d7da'))
        except:
            return colors.white
    
    table_style = TableStyle([
        ('TEXTCOLOR', (0,0), (-1,0), colors.white),
        ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),
        ('BACKGROUND', (0,0), (-1,0), colors.HexColor('#1b4965')),
        ('ALIGN', (0,0), (-1,-1), 'CENTER'),
        ('FONTNAME', (0,-1), (-1,-1), 'Helvetica-Bold'),
        ('BACKGROUND', (0,-1), (-1,-1), colors.HexColor('#0d1b2a')),
        ('TEXTCOLOR', (0,-1), (-1,-1), colors.white),
        ('GRID', (0,0), (-1,-1), 0.5, colors.grey),
        ('ROWBACKGROUNDS', (0,1), (-1,-2), [colors.whitesmoke, colors.white]),
    ])
    
    for i in range(1, len(score_data)-1):
        table_style.add('BACKGROUND', (1, i), (1, i), score_color(score_data[i][1]))
    
    tbl = Table(score_data, colWidths=[2.2*inch, 1.3*inch, 1.0*inch, 2.5*inch])
    tbl.setStyle(table_style)
    story.append(tbl)
    
    #  3. SPEAKING 
    section('3. Speaking Analysis')
    kv('Pace Classification', report.speaking_pace)
    kv('Words Per Minute', f'{report.total_words / max(report.duration_sec/60, 0.1):.1f}')
    if report.speaking_pace == 'Ideal':
        story.append(Paragraph('Your pace falls within the recommended 110–160 WPM range for professional interviews.', body))
    elif report.speaking_pace == 'Too Slow':
        story.append(Paragraph('Your pace is below 110 WPM. Consider increasing energy to maintain interviewer engagement.', body))
    else:
        story.append(Paragraph('Your pace exceeds 160 WPM. Slowing down will improve clarity and retention.', body))
    
    #  4. FILLERS 
    section('4. Filler Word Analysis')
    kv('Total Filler Words', filler.total_fillers)
    kv('Filler Ratio', f'{filler.filler_ratio:.2f}%')
    kv('Severity', filler.severity)
    story.append(Paragraph(f'<i>Methodology note:</i> {filler.false_positive_risk}', small))
    if filler.per_filler_counts:
        top = ', '.join(f'{w} ({c}x)' for w, c in sorted(filler.per_filler_counts.items(), key=lambda x: x[1], reverse=True)[:5])
        kv('Most Common', top)
    
    #  5. GAZE 
    section('5. Eye Contact & Gaze Analysis')
    valid_gaze = [g for g in gaze_frames if g.face_detected]
    if valid_gaze:
        looking_pct = (sum(1 for g in valid_gaze if g.looking_at_camera) / len(valid_gaze)) * 100
        kv('Frames Analyzed', len(gaze_frames))
        kv('Face Detected Rate', f'{len(valid_gaze)/len(gaze_frames)*100:.1f}%')
        kv('Looking at Camera', f'{looking_pct:.1f}%')
        kv('Average Gaze Score', f'{np.mean([g.gaze_score for g in valid_gaze]):.3f} (0-1)')
    else:
        story.append(Paragraph('No faces detected in sampled frames. Ensure good lighting and camera positioning.', body))
    story.append(Paragraph('<i>Methodology:</i> MediaPipe Face Mesh (468 landmarks) estimates iris position relative to eye center. Not a mind-reading tool.', small))
    
    #  6. EXPRESSIONS 
    section('6. Expression Analysis')
    valid_exp = [e for e in emotions if e.expression != 'no_detection']
    if valid_exp:
        exp_counts = Counter(e.expression for e in valid_exp)
        total = len(valid_exp)
        for expr, count in exp_counts.most_common():
            pct = (count / total) * 100
            story.append(Paragraph(f'• {expr.replace("_", " ").title()}: {pct:.1f}%', body))
    story.append(Paragraph('<i>Note:</i> These are facial expression classifications, not true emotional states. A furrowed brow may indicate concentration, not anger.', small))
    
    #  7. VOICE 
    section('7. Voice Feature Analysis')
    kv('Vocal Presence Score', f'{voice.confidence_score:.1f} — {voice.confidence_label}')
    kv('Energy', f'{voice.energy:.1f}')
    kv('Consistency', f'{voice.consistency:.1f}')
    kv('Speaking Ratio', f'{voice.speaking_ratio:.1f}%')
    kv('Pause Quality', f'{voice.pause_quality:.1f}')
    kv('Pitch Variability', f'{voice.pitch_variability:.1f}')
    kv('Spectral Clarity Proxy', f'{voice.clarity_proxy:.1f}')
    story.append(Paragraph('<i>All voice metrics are acoustic proxies. They measure signal properties, not personality traits.</i>', small))
    
    #  8. CONTENT 
    section('8. Content Quality')
    kv('Clarity Score', content.clarity_score)
    kv('Relevance Score', content.relevance_score)
    kv('Structure Score', content.structure_score)
    kv('Tech Vocab Score', content.tech_vocab_score)
    kv('Content Quality', content.content_quality)
    kv('Avg Sentence Length', f'{content.avg_sentence_length} words')
    if content.keywords:
        story.append(Paragraph(f'<b>Keywords Detected:</b> {", ".join(content.keywords[:15])}', body))
    story.append(Paragraph('<i>Keyword scoring is capped to prevent gaming. Depth &gt; density.</i>', small))
    
    #  9. SENTIMENT 
    section('9. Sentiment Snapshot')
    kv('Label', sentiment['label'])
    kv('Confidence', f'{sentiment["confidence"]:.3f}')
    story.append(Paragraph(sentiment['note'], small))
    
    #  10. STRENGTHS 
    section('10. Strengths')
    for s in report.strengths:
        story.append(Paragraph(f'✓ {s}', body))
    
    #  11. IMPROVEMENTS 
    section('11. Areas for Improvement')
    for imp in report.improvements:
        story.append(Paragraph(f'→ {imp}', body))
    
    #  12. TRANSCRIPT 
    section('12. Transcript Excerpt')
    story.append(Paragraph(report.transcript_snippet[:3000] + ' ...', small))
    
    doc.build(story)
    print(f'PDF saved: {output_path}')
    return output_path

## Cell 16: Main Analysis Pipeline

In [ ]:
from IPython.display import FileLink

def run_analysis(b):
    global video_path
    if video_path is None or not video_path.exists():
        show_status('Please upload a video first.', 'error')
        return
    
    try:
        # 1. Audio
        show_status('Step 1/7: Extracting audio...', 'analyze')
        audio_path = extract_audio(video_path)
        
        # 2. Frames
        show_status('Step 2/7: Sampling video frames...', 'analyze')
        frames, timestamps = sample_frames(video_path, target_fps=0.5, max_frames=240)
        
        # 3. Transcription
        show_status('Step 3/7: Transcribing speech...', 'analyze')
        if audio_path:
            transcript = transcribe_audio(audio_path)
        else:
            transcript = TranscriptResult(text='[No audio extracted]', duration_sec=timestamps[-1] if timestamps else 0, wpm=0.0)
        
        # 4. Gaze (MediaPipe)
        show_status('Step 4/7: Estimating gaze direction...', 'analyze')
        gaze_frames, avg_gaze = analyze_gaze(frames, timestamps)
        
        # 5. Expressions
        show_status('Step 5/7: Analyzing facial expressions...', 'analyze')
        emotions = analyze_expressions(frames, timestamps)
        
        # 6. Voice & Fillers & Content
        show_status('Step 6/7: Analyzing voice & content...', 'analyze')
        voice = analyze_voice(audio_path) if audio_path else VoiceFeatures(
            confidence_score=0.0, confidence_label='No Audio',
            energy=0.0, consistency=0.0, speaking_ratio=0.0, pause_quality=0.0,
            pitch_variability=0.0, expression_index=0.0, steadiness=0.0, clarity_proxy=0.0
        )
        filler = detect_fillers(transcript.text)
        content = analyze_content_quality(transcript.text)
        sentiment = analyze_sentiment(transcript.text) if transcript.text else {'label': 'N/A', 'confidence': 0.0, 'note': 'No transcript'}
        
        # 7. Score & PDF
        show_status('Step 7/7: Computing scores & generating report...', 'analyze')
        report = compute_scores(transcript, gaze_frames, voice, filler, content, emotions)
        pdf_path = generate_pdf(report, gaze_frames, emotions, filler, content, voice, sentiment)
        
        # Display results
        clear_output(wait=True)
        display(HTML(f'''
        <div style='padding:20px;border-radius:16px;background:linear-gradient(135deg,#1b4965,#0d1b2a);color:white;'>
            <h2>✅ Analysis Complete</h2>
            <div style='font-size:32px;font-weight:bold;margin:10px 0;'>{report.overall_score:.1f}<span style='font-size:16px;'>/100</span></div>
            <div style='font-size:18px;margin-bottom:16px;'>Verdict: <b>{report.verdict}</b></div>
            <div style='display:flex;gap:12px;flex-wrap:wrap;'>
                <div style='background:rgba(255,255,255,0.15);padding:8px 16px;border-radius:8px;'>Comm: {report.communication:.1f}</div>
                <div style='background:rgba(255,255,255,0.15);padding:8px 16px;border-radius:8px;'>Conf: {report.confidence:.1f}</div>
                <div style='background:rgba(255,255,255,0.15);padding:8px 16px;border-radius:8px;'>Content: {report.content_quality:.1f}</div>
                <div style='background:rgba(255,255,255,0.15);padding:8px 16px;border-radius:8px;'>Eye: {report.eye_contact_score:.1f}</div>
                <div style='background:rgba(255,255,255,0.15);padding:8px 16px;border-radius:8px;'>Stability: {report.emotion_stability:.1f}</div>
            </div>
        </div>
        '''))
        
        display(FileLink(str(pdf_path), result_html_prefix='📄 Download Report: '))
        
        # Print honest console summary
        print('\n' + '='*60)
        print('HONEST SELF-ASSESSMENT SUMMARY')
        print('='*60)
        print(f'Overall: {report.overall_score:.1f}/100 ({report.verdict})')
        print(f'Pace: {report.speaking_pace}')
        print(f'Fillers: {filler.filler_ratio:.2f}% ({filler.severity})')
        print(f'Gaze: {avg_gaze*100:.1f}% avg alignment')
        print(f'Technical keywords: {len(content.keywords)}')
        print(f'\nTop 3 strengths:')
        for s in report.strengths[:3]:
            print(f'  • {s}')
        print(f'\nTop 3 improvements:')
        for i in report.improvements[:3]:
            print(f'  • {i}')
        print('='*60)
        
    except Exception as e:
        show_error(e, 'Main Pipeline')
        show_status('Analysis failed. Check error details above.', 'error')

analyze_btn.on_click(run_analysis)
print("Pipeline ready. Upload a video and click 'Analyze Video'.")